In [21]:
from langchain_openai import AzureOpenAIEmbeddings
from neo4j import GraphDatabase
from langchain_community.graphs import Neo4jGraph
import os

import warnings
warnings.filterwarnings('ignore')

I have tried looking into text-embedding-3-large and text-embedding-3-large embedding models but the length of each embedding vector is 4096 and 1536, so long that would take up 500MB to 1GB of memory in the neo4j graph. 

I'm now going to use SBERT embedding model because the embedding vector is much smaller, i.e. len = 384

In [22]:
from sentence_transformers import SentenceTransformer

In [23]:
SBERT_embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

In [24]:
os.environ["NEO4J_URI"] = "bolt://localhost:7687"
os.environ["NEO4J_USERNAME"] = "neo4j"
os.environ["NEO4J_PASSWORD"] = "password"

In [25]:
graph = Neo4jGraph()

In [26]:
cypher = """MATCH (n) RETURN n.proposition, n.locution, n.prediction_index
"""

result = graph.query(cypher)
result[:10]

[{'n.proposition': 'even if some children do get back to school before the end of the summer term, their experience is more likely to be about social distancing and hygiene rather than anything of any educational value',
  'n.locution': 'Even if some children do get back to school before the end of the summer term, their experience is more likely to be about social distancing and hygiene rather than anything of any educational value',
  'n.prediction_index': 0},
 {'n.proposition': 'xxx is what the panel can suggest for a whole generation of school children who are now missing nearly half an academic year and face months of distrust about social distancing and social contact and the face of further disruption',
  'n.locution': 'What can the panel suggest for a whole generation of school children who are now missing nearly half an academic year and face months of distrust about social distancing and social contact and the face of further disruption',
  'n.prediction_index': 1},
 {'n.prop

In [27]:
proposition = result[0]["n.proposition"]
print(proposition)

even if some children do get back to school before the end of the summer term, their experience is more likely to be about social distancing and hygiene rather than anything of any educational value


In [28]:
proposition_embedding_vector = SBERT_embedding_model.encode(proposition)
proposition_embedding_vector[:10]

array([ 0.03200769,  0.07649335,  0.08485737,  0.07419854,  0.12188505,
        0.01508242, -0.05564725, -0.08257454, -0.04321716,  0.02995576],
      dtype=float32)

In [29]:
len(proposition_embedding_vector)

384

In [30]:
locution = result[0]["n.locution"]
print(locution)

Even if some children do get back to school before the end of the summer term, their experience is more likely to be about social distancing and hygiene rather than anything of any educational value


In [31]:
locution_embedding_vector = SBERT_embedding_model.encode(locution)
locution_embedding_vector[:10]

array([ 0.03200769,  0.07649335,  0.08485737,  0.07419854,  0.12188505,
        0.01508242, -0.05564725, -0.08257454, -0.04321716,  0.02995576],
      dtype=float32)

In [32]:
concatenated_loc_and_prop = locution + " " + proposition
print(concatenated_loc_and_prop)

Even if some children do get back to school before the end of the summer term, their experience is more likely to be about social distancing and hygiene rather than anything of any educational value even if some children do get back to school before the end of the summer term, their experience is more likely to be about social distancing and hygiene rather than anything of any educational value


In [33]:
concatenated_loc_and_prop_embedding_vector = SBERT_embedding_model.encode(concatenated_loc_and_prop)
concatenated_loc_and_prop_embedding_vector[:10]

array([ 0.01826683,  0.07018496,  0.09295951,  0.07481697,  0.13671255,
        0.02560251, -0.03759161, -0.08110321, -0.02330314,  0.02594815],
      dtype=float32)

In [34]:
len(concatenated_loc_and_prop_embedding_vector)

384

In [35]:
from numpy import dot
from numpy.linalg import norm

In [36]:
embedding_vectors = [proposition_embedding_vector,
                     locution_embedding_vector,
                     concatenated_loc_and_prop_embedding_vector]


for i_vec in embedding_vectors:
    for j_vec in embedding_vectors:

        cos_sim = dot(i_vec, j_vec)/(norm(i_vec)*norm(j_vec))
        print(cos_sim)

    print("-----------------------")

1.0000001
1.0000001
0.97487324
-----------------------
1.0000001
1.0000001
0.97487324
-----------------------
0.97487324
0.97487324
1.0
-----------------------


In [37]:
len(result)

23228

In [18]:
from neo4j import GraphDatabase

data_base_connection = GraphDatabase.driver(uri = "bolt://localhost:7687", auth=("neo4j", "password"))
session = data_base_connection.session()  

In [19]:
embedding_model_name="embedding_from_all_MiniLM_L6_v2"

Concatenated the locution and proposition, computed SBERT embedding and add embedding vector to each node in neo4j graph

In [20]:
for res in result:
    #print(res)
    proposition = res["n.proposition"]
    proposition_embedding_vector = SBERT_embedding_model.encode(proposition)
    locution = res["n.locution"]
    locution_embedding_vector = SBERT_embedding_model.encode(locution)
    concat_loc_and_prop = proposition + " " + locution
    concat_loc_and_prop_embedding_vector = SBERT_embedding_model.encode(concat_loc_and_prop)
    pred_index = res["n.prediction_index"]

    cypher = f"""MATCH (n)
    WHERE n.proposition = "{proposition}" AND n.locution = "{locution}" AND n.prediction_index = {pred_index}
    SET n.loc_and_prop_concat_{embedding_model_name} = {concat_loc_and_prop_embedding_vector.tolist()}
"""
    session.run(cypher)
